In [19]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
import numpy as np
import random

def load_finbert_model():
    """Load the FinBERT model"""
    print("Loading FinBERT model...")
    try:
        tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
        model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        model.eval()
        print(f"Model loaded successfully on {device}")
        return tokenizer, model, device
    except Exception as e:
        print(f"Error loading model: {e}")
        raise

class HeterogeneousFinBERTAgent:
    """Heterogeneous News Trader powered by ProsusAI/finbert."""
    
    # Class-level attribute: Load the model ONCE for all 50 agents to share
    nlp = pipeline("sentiment-analysis", model="ProsusAI/finbert")

    def __init__(self, agent_id):
        self.agent_id = agent_id
        
        # HETEROGENEITY 1: Different Risk Tolerances
        # Some agents are trigger-happy (0.55), others are highly conservative (0.85)
        self.confidence_threshold = random.uniform(0.55, 0.85) 
        
        # HETEROGENEITY 2: Different Fund Capital Sizes
        # Agents trade fundamentally different baseline volumes
        self.base_qty = random.randint(50, 500) 

    def analyze_news_and_trade(self, news_headline):
        """Passes text to FinBERT and applies agent-specific risk parameters."""
        
        # 1. The deterministic model inference
        result = self.nlp(news_headline)[0]
        sentiment = result['label']
        confidence = result['score']
        
        # 2. The heterogeneous decision logic
        action = None
        if sentiment == "positive" and confidence > self.confidence_threshold:
            action = "buy"
        elif sentiment == "negative" and confidence > self.confidence_threshold:
            action = "sell"
            
        # 3. Output the structured execution
        if action:
            # HETEROGENEITY 3: Execution Noise
            # Traders rarely execute perfectly round lots based purely on confidence
            execution_noise = random.uniform(0.9, 1.1) 
            qty = int(self.base_qty * confidence * execution_noise)
            
            print(f"[{self.agent_id}] Thresh: {self.confidence_threshold:.2f} | Action: {action.upper()} {qty} shares")
            
            return {
                "agent_id": self.agent_id,
                "type": "MARKET",
                "side": action,
                "qty": qty
            }
            
        # Agent decides to hold if the model's confidence doesn't meet its specific threshold
        print(f"[{self.agent_id}] Thresh: {self.confidence_threshold:.2f} | Action: HOLD (Confidence {confidence:.2f} too low)")
        return None

In [ ]:
# TESTING the LLM class
# Initialize 10 heterogeneous agents
finbert_traders = [HeterogeneousFinBERTAgent(f"News_Trader_{i}") for i in range(10)]

# The Exogenous Shock [cite: 50, 405]
breaking_news = "Federal Reserve announces unexpected emergency 50 bps rate hike due to banking sector stress"

# All agents process the same semantic data and output correlated actions [cite: 5, 37]
toxic_order_flow = []
for agent in finbert_traders:
    order = agent.analyze_news_and_trade(breaking_news)
    print(order)
    if order:
        toxic_order_flow.append(order)

[News_Trader_0] Thresh: 0.84 | Action: HOLD (Confidence 0.79 too low)
None
[News_Trader_1] Thresh: 0.74 | Action: SELL 86 shares
{'agent_id': 'News_Trader_1', 'type': 'MARKET', 'side': 'sell', 'qty': 86}
[News_Trader_2] Thresh: 0.62 | Action: SELL 156 shares
{'agent_id': 'News_Trader_2', 'type': 'MARKET', 'side': 'sell', 'qty': 156}
[News_Trader_3] Thresh: 0.81 | Action: HOLD (Confidence 0.79 too low)
None
[News_Trader_4] Thresh: 0.57 | Action: SELL 198 shares
{'agent_id': 'News_Trader_4', 'type': 'MARKET', 'side': 'sell', 'qty': 198}
[News_Trader_5] Thresh: 0.84 | Action: HOLD (Confidence 0.79 too low)
None
[News_Trader_6] Thresh: 0.80 | Action: HOLD (Confidence 0.79 too low)
None
[News_Trader_7] Thresh: 0.72 | Action: SELL 237 shares
{'agent_id': 'News_Trader_7', 'type': 'MARKET', 'side': 'sell', 'qty': 237}
[News_Trader_8] Thresh: 0.70 | Action: SELL 251 shares
{'agent_id': 'News_Trader_8', 'type': 'MARKET', 'side': 'sell', 'qty': 251}
[News_Trader_9] Thresh: 0.73 | Action: SELL 396